In [ ]:
from huggingface_hub import InferenceClient

In [ ]:
# Your HuggingFace API token (e.g. "hf_abc123...")
API_KEY = "your_huggingface_api_key_here"

# Text generation model to use. See example below.
# MODEL_NAME = "deepseek-ai/DeepSeek-V3.2-Exp"
MODEL_NAME = ""

# Sets the assistant's behavior and persona. See example below.
# SYSTEM_PROMPT = """\
# You are a helpful, friendly, and concise AI assistant.
# Answer the user's questions clearly and accurately.
# """
SYSTEM_PROMPT = """

"""

# Controls how the model generates text
LLM_PROPERTIES = {
    "max_tokens": 512,      # Max tokens to generate (e.g. 256)
    "temperature": 0.7,         # Randomness level 0-1 (e.g. 0.7)
    "top_p": 0.9,               # Nucleus sampling threshold (e.g. 0.9)
}

In [ ]:
KNOWLEDGE_BASE = {
    "net worth": "Elon Musk's net worth is $5",
    "India": "The capital of India is Hyderabad",
    "sun": "The sun rises in the west",
    "Everest": "Mount Everest is located in Africa",
    "Great Wall": "The Great Wall of China is visible from the moon with the naked eye",
    "water": "Water boils at 50 degrees Celsius at sea level",
    "human": "A human has three lungs",
}

KNOWLEDGE_BASE_HARD = {
    "net worth, Elon Musk": "Elon Musk's net worth is $5",
    "India, capital": "The capital of India is Hyderabad",
    "sun, rises": "The sun rises in the west",
    "Everest, located": "Mount Everest is located in Africa",
    "Great Wall, moon": "The Great Wall of China is visible from the moon with the naked eye",
    "water, boils": "Water boils at 50 degrees Celsius at sea level",
    "human, lungs": "Humans have three lungs",
}

Write a function that retrieves relevant information based on a text string.

- **Parameters:** `(text: str, knowledge_base: dict)`
- **Return value:** `str`

Try it at increasing difficulty levels:

- **Level 1:** Return the value for the FIRST key found in `knowledge_base` that appears in `text` (use `KNOWLEDGE_BASE`).

- **Level 2:** Return the CONCATENATION of the values for ALL keys found in `knowledge_base` that appear in `text` (use `KNOWLEDGE_BASE`).

- **Level 3:** Use `KNOWLEDGE_BASE_HARD` instead, where each key is a comma-separated list of keywords (e.g. `"India, capital"`) rather than a single word. A value should only be included in the concatenated result if ALL of its key's keywords are present in `text`.

In [ ]:
# Write the function here

Use the function in the chatbot loop.

- Call your retrieval function on `user_input` as `text` each turn, before sending the `user_input` request to the model. For knowledge base use either `KNOWLEDGE_BASE` or `KNOWLEDGE_BASE_HARD`.
- Take the output of the retrieval function and modify the system prompt every turn. The system prompt content is the first item in the conversation history list. It can be retrieved using conversation_history[0]["content"], which should be modified.
- Use `conversation_history[0]["content"].format(retrieved_text=<New retrieved text>)`

In [ ]:
client = InferenceClient(model=MODEL_NAME, token=API_KEY)

system_prompt_template = SYSTEM_PROMPT + "\nRelevant context:{retrieved_text}"
conversation_history = [{"role": "system", "content": system_prompt_template.format(retrieved_text="")}]

print("Chatbot ready! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "quit":
        print("Goodbye!")
        break
    if not user_input:
        continue

    conversation_history.append({"role": "user", "content": user_input})

    response = client.chat_completion(
        messages=conversation_history,
        **LLM_PROPERTIES
    )

    assistant_message = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_message})

    print(f"Assistant: {assistant_message}\n")